In [1]:
import os  # 작업 디렉터리 변경(os.chdir)에 사용
import pathlib  # 경로를 객체로 다루기 위한 모듈
import sys  # 모듈 검색 경로(sys.path)를 조작하기 위해 사용

here = pathlib.Path.cwd()  # 현재 작업 디렉터리 (노트북을 처음 열면 sandbox/W7/Fri)
ROOT = here.parents[2] if here.name == "Fri" else here  # Thu 폴더면 세 단계 위(프로젝트 루트), 이미 루트로 옮겨졌으면 그대로 사용
os.chdir(ROOT)  # 이후 상대 경로가 프로젝트 루트 기준이 되도록 작업 디렉터리 이동
SANDBOX = ROOT / "sandbox" / "w7" / "Fri"  # 오늘 실습 폴더 경로

print("프로젝트 루트  :", ROOT)  # 루트를 제대로 잡았는지 눈으로 확인

if str(ROOT / 'backend') not in sys.path:  # backend 폴더가 아직 import 경로에 없으면
    sys.path.insert(0, str(ROOT / 'backend'))  # 맨 앞에 추가 → `import app...` 가 가능해짐 (커널 재시작 시 사라지므로 다시 실행)

SAMPLES = ROOT / "sandbox" / "w5" / "Mon" / "samples"  # W5 월요일에 만든 샘플 문서 폴더
FORMATS = SAMPLES / "_formats"  # 포맷별 샘플 파일이 모인 하위 폴더

프로젝트 루트  : d:\hanwha-agent


In [2]:
from sqlalchemy import select
from app.core.config import get_settings
from app.db.session import session_scope
from app.integrations.factory import get_embedder
from app.rag import store
from app.rag.retriever import expand_queries
settings = get_settings()
QUESTION = '1~2급 직원의 숙박비 상한은 얼마인가요?'

def candidates_for(question: str) -> list[dict]:
    embedder = get_embedder()
    with session_scope() as session:
        seen: dict[int, dict] = {}
        for one in expand_queries(question, settings.multi_query_n):
            for row in store.vector_search(session, embedder.embed_query(one), limit=settings.retrieval_top_k):
                seen.setdefault(row['chunk_id'], row)
        vector_rows = sorted(seen.values(), key=lambda r: r['score'], reverse=True)
        keyword_rows = store.keyword_search(session, question, limit=settings.retrieval_top_k)
    return store.fuse(vector_rows, keyword_rows)[:settings.retrieval_top_k]
rows = candidates_for(QUESTION)
for (n, row) in enumerate(rows, 1):
    if n == settings.rerank_top_n + 1:
        print(f"{'─' * 27} 여기서 잘립니다 (rerank_top_n = {settings.rerank_top_n})")
    print(f"{n:>2}  {row['doc_id']}  {row['version']}  {row['kind']:<2}  {row['locator'][:22]:<22} {row['score']:.3f}")

12:08:38 INFO     sentence_transformers.base.model: No device provided, using cpu
12:08:38 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

12:08:40 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: models\bge-m3/1024 차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 1  DOC-HR-014  v2.0  표   표2                     0.714
 2  DOC-HR-014  v1.1  표   표2                     0.690
 3  DOC-HR-014  v1.1  조항  제3장 제11조(숙박비) · 별표1    0.648
─────────────────────────── 여기서 잘립니다 (rerank_top_n = 3)
 4  DOC-HR-014  v2.0  조항  제3장 제2절 제14조(숙박비) · 별표 0.691
 5  DOC-HR-014  v2.0  표   표3                     0.647
 6  DOC-HR-014  v2.0  조항  제3장 제1절 제11조(일비) · 별표1 0.601
 7  DOC-HR-014  v1.1  조항  제3장 제8조(일비) · 별표1      0.594
 8  DOC-HR-014  v2.0  조항  제3장 제2절 제15조(지역 구분) ·  0.590
 9  DOC-HR-014  v2.0  조항  제3장 제2절 제16조(숙박비의 예외)  0.659
10  DOC-HR-014  v2.0  조항  제3장 제1절 제12조(식비) · 별표1 0.653
11  DOC-HR-014  v1.1  조항  제1장 제4조(여비의 종류) · 별표1  0.000
12  DOC-HR-014  v1.1  조항  제3장 제9조(식비)            0.643


In [3]:
from app.rag.reranker import rerank, score as lexical_score

before = [(r['doc_id'], r['locator'][:12], r['score'], lexical_score(QUESTION, r['text']), 0.03 if r['kind'] == '표' else 0.0) for r in rows]

after = rerank(QUESTION, rows, top_n=len(rows))

print('  [리랭킹 전]                                  [리랭킹 후]')
for (n, (left, right)) in enumerate(zip(before, after), 1):
    (doc_id, locator, vec, lex, bonus) = left
    print(f"{n:>2} {doc_id} {locator:<12} 벡터 {vec:.3f}   {n:>2} {right['doc_id']} {right['locator'][:12]:<12} rerank {right['rerank_score']:.3f}  (벡터 {right['score']:.3f})")


12:08:43 INFO     sentence_transformers.base.model: No device provided, using cpu
12:08:43 INFO     sentence_transformers.base.model: No modules.json found for models\bge-reranker-v2-m3, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

12:08:47 INFO     app.integrations.local_rerank: 로컬 리랭커 준비 : models\bge-reranker-v2-m3, 최대 512토큰
  [리랭킹 전]                                  [리랭킹 후]
 1 DOC-HR-014 표2           벡터 0.714    1 DOC-HR-014 표2           rerank 0.893  (벡터 0.690)
 2 DOC-HR-014 표2           벡터 0.690    2 DOC-HR-014 표2           rerank 0.868  (벡터 0.714)
 3 DOC-HR-014 제3장 제11조(숙박비 벡터 0.648    3 DOC-HR-014 제3장 제2절 제14조 rerank 0.516  (벡터 0.691)
 4 DOC-HR-014 제3장 제2절 제14조 벡터 0.691    4 DOC-HR-014 표3           rerank 0.494  (벡터 0.647)
 5 DOC-HR-014 표3           벡터 0.647    5 DOC-HR-014 제3장 제2절 제16조 rerank 0.240  (벡터 0.659)
 6 DOC-HR-014 제3장 제1절 제11조 벡터 0.601    6 DOC-HR-014 제3장 제11조(숙박비 rerank 0.150  (벡터 0.648)
 7 DOC-HR-014 제3장 제8조(일비)  벡터 0.594    7 DOC-HR-014 제3장 제1절 제11조 rerank 0.009  (벡터 0.601)
 8 DOC-HR-014 제3장 제2절 제15조 벡터 0.590    8 DOC-HR-014 제3장 제8조(일비)  rerank 0.008  (벡터 0.594)
 9 DOC-HR-014 제3장 제2절 제16조 벡터 0.659    9 DOC-HR-014 제3장 제1절 제12조 rerank 0.003  (벡터 0.653)
10 DOC-HR-014 제3장 제1절 제12조 벡터 0.653   10 D

In [4]:
import os
import huggingface_hub.constants as hf_constants
from huggingface_hub import snapshot_download

# 앞 셀에서 임베딩 모델을 열 때 LocalEmbedder 가 오프라인 모드(HF_HUB_OFFLINE=1)를 켜 두므로, 다운로드 전에 해제
os.environ["HF_HUB_OFFLINE"] = "0"
hf_constants.HF_HUB_OFFLINE = False

MODELS = ROOT / "models"   # bge-m3 와 같은 폴더 (.gitignore 대상)
TARGET = MODELS / "bge-reranker-v2-m3"
TARGET.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="BAAI/bge-reranker-v2-m3",
    local_dir=str(TARGET),
    allow_patterns=["*.json", "*.safetensors", "sentencepiece.bpe.model"],
)

for p in sorted(TARGET.iterdir()):
    if p.is_file():
        print(f"{p.name:<28} {p.stat().st_size / 1024**2:>10,.1f} MB")

size_gb = (TARGET / "model.safetensors").stat().st_size / 1024**3
print(f"\nmodel.safetensors = {size_gb:.2f} GB →",
      "정상" if size_gb > 2.0 else "덜 받았습니다 — 이 셀을 다시 실행하세요")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

config.json                         0.0 MB
model.safetensors               2,165.9 MB
sentencepiece.bpe.model             4.8 MB
special_tokens_map.json             0.0 MB
tokenizer.json                     16.3 MB
tokenizer_config.json               0.0 MB

model.safetensors = 2.12 GB → 정상


In [5]:
import importlib
import time
from app.core import config
from app.integrations import factory, ports
from app.rag import reranker

for module in (config, ports, factory, reranker):
    importlib.reload(module)
    
config.get_settings.cache_clear()
print('RERANKER :', config.get_settings().reranker)

start = time.perf_counter()
model = factory.get_cross_encoder()
print(f'모델을 열었습니다 ({time.perf_counter() - start:.1f}s)')

QUESTION = '해외 출장 시 비즈니스석 탑승 기준이 어떻게 되나요?'
PIECES = ['제7조(출장 신청) ① 출장을 하려는 임직원은 출장 개시일 3일 전까지 출장 신청서를 제출한다.', '제14조(숙박비) ② 광역시 지역의 숙박비는 1박당 70,000원을 상한으로 한다.']
start = time.perf_counter()
scores = model.score_pairs(QUESTION, PIECES)

for piece, s in zip(PIECES, scores):
    print(f'{s:.4f}  {piece[:24]}')
print(f'두 쌍 {time.perf_counter() - start:.2f}s · 합격선 {config.get_settings().rerank_threshold}')


RERANKER : cross_encoder
12:09:01 INFO     sentence_transformers.base.model: No device provided, using cpu
12:09:01 INFO     sentence_transformers.base.model: No modules.json found for models\bge-reranker-v2-m3, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

12:09:05 INFO     app.integrations.local_rerank: 로컬 리랭커 준비 : models\bge-reranker-v2-m3, 최대 512토큰
모델을 열었습니다 (4.4s)
0.0004  제7조(출장 신청) ① 출장을 하려는 임직원
0.0000  제14조(숙박비) ② 광역시 지역의 숙박비는
두 쌍 0.43s · 합격선 0.2


In [ ]:
import subprocess

CHECK=(
    "from app.models.document import VERSION_CURRENT\n"
    "from app.rag.policy import CURRENT\n"
    "print(f\"VERSION_CURRENT='{VERSION_CURRENT}'\")\n"
    "print(f\"policy.CURRENT='{CURRENT}'\")\n"
    "print('두 값이 같습니다' if VERSION_CURRENT==CURRENT else '두 값이 다릅니다')\n"
)

done = subprocess.run([sys.executable, '-c', CHECK], cwd=ROOT, capture_output=True,
                      text=True, env={**os.environ, 'PYTHONPATH':str(ROOT/'backend')})

if done.returncode==0:
    print(done.stdout.strip())
else:
    print('VERSION_CURRENT가 작성되지 않았습니다.')

In [9]:
from app.rag import policy as pol
VIEWER = {'name': '김민준', 'dept_id': 'INFRA2', 'clearance': '일반'}
for label, question in (('[기본]', QUESTION), ('[「개정 전」]', f'개정 전 {QUESTION}')):
    found = candidates_for(question)
    kept, blocked = pol.filter_candidates(found, user=VIEWER, question=question)
    print(f'{label:<12} 후보 {len(found)}건 → 통과 {len(kept)}건')
    for one in list(dict.fromkeys(blocked))[:4]:
        print(f'   거른 사유 : {one}')

12:45:49 INFO     sentence_transformers.base.model: No device provided, using cpu
12:45:49 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

12:45:51 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: models\bge-m3/1024 차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

[기본]         후보 12건 → 통과 6건
   거른 사유 : 국내출장 여비 규정 v1.1 - 만료 버전
   거른 사유 : 법인 카드 사용 지침 v1.4 - 만료 버전


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[「개정 전」]     후보 12건 → 통과 12건
